<a href="https://colab.research.google.com/github/safarashwan2005/Default-optimized-/blob/randomizing/default_optimized_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import warnings
import numpy as np
import scipy.sparse as sp
from scipy.sparse import kron as sp_kron
import cupy as cp
import cupyx.scipy.sparse as cp_sp
import time
import gc
start_time = time.perf_counter()

warnings.filterwarnings("ignore", category=sp.SparseEfficiencyWarning)



class NVCenterSystemBaseGPU:

    def __init__(self, params):
        self.N = params['N']
        self.D = params['D']
        self.omega_0 = params['omega_0']
        self.gamma_e = params['gamma_e']
        self.Bz = params['Bz']
        self.zeta = params['zeta']
        self.g = params['g']

        # Single-spin operators constructed on CPU to avoid GPU VRAM exhaustion during expansion
        self.I3 = sp.eye(3, dtype=complex, format='csr')
        self.Sz = sp.csr_matrix(
            np.array([[1, 0, 0], [0, 0, 0], [0, 0, -1]], dtype=complex)
        )
        self.Sx = sp.csr_matrix(
            (1 / np.sqrt(2))
            * np.array([[0, 1, 0], [1, 0, 1], [0, 1, 0]], dtype=complex)
        )
        self.Sy_prime = sp.csr_matrix(
            (1 / np.sqrt(2))
            * np.array([[0, -1j, 0], [1j, 0, -1j], [0, 1j, 0]], dtype=complex)
        )

    def _expand_operator_sparse(self, op, target_index):
        """Expands a 3x3 operator onto the N-spin Hilbert space in CPU system memory."""
        result = op if target_index == 0 else self.I3
        for i in range(1, self.N):
            current_op = op if i == target_index else self.I3
            result = sp_kron(result, current_op, format='csr')
        return result

    def get_control_hamiltonian(self):
        dim = 3**self.N
        H_control = sp.csr_matrix((dim, dim), dtype=complex)
        cos_z = np.cos(self.zeta)
        sin_z = np.sin(self.zeta)

        for j in range(self.N):
            Sx_j = self._expand_operator_sparse(self.Sx, j)
            Sy_p_j = self._expand_operator_sparse(self.Sy_prime, j)
            H_control = H_control + 0.5 * (cos_z * Sx_j + sin_z * Sy_p_j)

        return H_control

    def get_ground_state(self):
        ms_0 = np.array([0, 1, 0], dtype=complex)
        ground_state = ms_0
        for _ in range(1, self.N):
            ground_state = np.kron(ground_state, ms_0)
        return ground_state

    def _euler_solve_optimized(self, H_drift, H_control, u_array, y0, t_total, num_steps):
        # Executes this integration on GPU.
        dt = t_total / num_steps
        c = -1j
        psi = y0.copy()

        for i in range(num_steps):
            u_t = u_array[i]
            k1 = c * (H_drift.dot(psi) + u_t * H_control.dot(psi))
            psi += dt * k1

        return psi

    def evolve_state_optimized(self, initial_state, t_total, control_pulse, time_steps=2000):
        if not callable(control_pulse):
            raise TypeError("control_pulse must be a callable function (e.g., a lambda).")

        H_drift_cpu = self.get_drift_hamiltonian()
        H_control_cpu = self.get_control_hamiltonian()

        H_drift = cp_sp.csr_matrix(H_drift_cpu)
        H_control = cp_sp.csr_matrix(H_control_cpu)

        initial_state_gpu = cp.asarray(initial_state).reshape(-1, 1)

        # Generate timestamps for Euler (1 per step)
        euler_time_points = cp.linspace(0, t_total, time_steps, endpoint=False)

        t_np = cp.asnumpy(euler_time_points)
        u_array = cp.asarray(control_pulse(t_np))

        return self._euler_solve_optimized(
            H_drift, H_control, u_array, initial_state_gpu, t_total, num_steps=time_steps
        )
class NVCenterNearestNeighborGPU(NVCenterSystemBaseGPU):

    def get_drift_hamiltonian(self):
        dim = 3**self.N
        H_drift = sp.csr_matrix((dim, dim), dtype=complex)

        Sz_sq = self.Sz.dot(self.Sz)

        for j in range(self.N):
            Sz_j_sq = self._expand_operator_sparse(Sz_sq, j)
            Sz_j = self._expand_operator_sparse(self.Sz, j)

            term1 = (self.D[j] - self.omega_0) * Sz_j_sq
            term2 = self.gamma_e * self.Bz[j] * Sz_j
            H_drift = H_drift + (term1 - term2)

        for j in range(self.N - 1):
            Sz_j = self._expand_operator_sparse(self.Sz, j)
            Sz_j_next = self._expand_operator_sparse(self.Sz, j + 1)
            H_drift = H_drift + self.g[j] * Sz_j.dot(Sz_j_next)

        return H_drift


# =======================================
# Execution Block
# ==========================================
if __name__ == "__main__":
    NUM_RUNS = 10
    n_range = range(1, 9)
    times_per_N = {N: [] for N in n_range}
    norms_per_N = {N: [] for N in n_range}
    saved_states = {}          # used for graphd
    #---------------------------------------------------------------------
    # Warm-up (not timed): triggers one-time CUDA/cuSPARSE/CuPy setup
    _p = {'N': 2, 'D': [2870.0]*2, 'omega_0': 2870.0, 'gamma_e': 28000.0,
      'Bz': [1e-3]*2, 'zeta': np.pi/4, 'g': [0.1]}
    _w = NVCenterNearestNeighborGPU(_p)
    _w.evolve_state_optimized(_w.get_ground_state(), 1.0,
                         lambda t: 2.0*np.sin(np.pi*t), time_steps=200)
    cp.cuda.Stream.null.synchronize()
#------------------------------------------------------------------------------------------
        # ---- Randomized benchmarking setup ----
    SEED = 12345                                    # we can change this number to get a different set of random pulses
    rng = np.random.default_rng(SEED)
    steps_ladder = np.round(np.logspace(0, 4, NUM_RUNS)).astype(int)   # log-spaced from 1 to 10000
    run_log = []                                    # remembers what was used in every run
#----------------------------------------------------------------------------------------------
    for run in range(1, NUM_RUNS + 1):
        print(f"\n========================================")
        print(f"             STARTING RUN {run}/{NUM_RUNS}             ")
        print(f"========================================")


 # ------- Random pulse for this run (the same pulse is used for every N in this run) ----
        c1 = rng.uniform(0, 5)
        c2 = rng.uniform(0, 5)
        w1 = rng.uniform(0, 10 * np.pi)
        w2 = rng.uniform(0, 10 * np.pi)
        n_steps = int(steps_ladder[run - 1])
        lambda_pulse = lambda t: c1 * np.sin(w1 * t) + c2 * np.cos(w2 * t)
        pulse_text = f"u(t) = {c1:.4f}*sin({w1:.4f}*t) + {c2:.4f}*cos({w2:.4f}*t)"
        run_log.append((run, n_steps, c1, w1, c2, w2))
        print(f"RANDOM PULSE FOR THIS RUN: {pulse_text}")
        print(f"c1 = {c1:.4f}, w1 = {w1:.4f}, c2 = {c2:.4f}, w2 = {w2:.4f}, time steps = {n_steps}")
   #---------------------------------------------------------------------------------------------------
        for N_test in n_range:
         #-----------------------------------------
            cp.cuda.Stream.null.synchronize()
            iter_start_time = time.perf_counter()
         #-----------------------------------------
            print(f"\n{'='*40}")
            print(f" [RUN {run}/{NUM_RUNS}] SIMULATING FOR N = {N_test} NV CENTERS ")
            print(f"{'='*40}")


            test_params = {
                'N': N_test,
                'D': [2870.0] * N_test,
                'omega_0': 2870.0,
                'gamma_e': 28000.0,
                'Bz': [1e-3] * N_test,
                'zeta': np.pi / 4,
                'g': [0.1] * (N_test - 1),
            }

            print("Building Sparse Hamiltonians on CPU...")
            nv_chain = NVCenterNearestNeighborGPU(test_params)
            initial_state_cpu = nv_chain.get_ground_state()
            t_pulse = 1.0

            print("--- Testing Lambda Function Pulse ---")
            print(f"Pulse used : {pulse_text}")
            print(f"Time steps : {n_steps}")

            final_state_lambda = nv_chain.evolve_state_optimized(
                initial_state=initial_state_cpu,
                t_total=t_pulse,
                control_pulse=lambda_pulse,
                time_steps=n_steps,
            )

            norm_lambda = cp.asnumpy(cp.linalg.norm(final_state_lambda))
            print("Norm check (Lambda):", norm_lambda)
            norms_per_N[N_test].append(float(norm_lambda))

#--------------------------------------------------------------
            cp.cuda.Stream.null.synchronize()
            iter_end_time = time.perf_counter()
            elapsed_time = iter_end_time - iter_start_time


            times_per_N[N_test].append(elapsed_time)
            if N_test <= 4:
                saved_states[(N_test, run)] = cp.asnumpy(final_state_lambda).ravel()
            print(f"\n[TIME TAKEN FOR N = {N_test} (Run {run})]: {elapsed_time:.4f} seconds")

            # Free GPU memory and CPU garbage collection
            del nv_chain, initial_state_cpu, final_state_lambda
            gc.collect()
            cp.get_default_memory_pool().free_all_blocks()
#---------------------------------------------------------------------------------------------------------------------------

    end_time = time.perf_counter()

    print(f"\n{'='*50}")
    print(f"TOTAL EXECUTION TIME (ALL {NUM_RUNS} RUNS): {end_time - start_time:.4f} seconds")
    print(f"{'='*50}")

    print(f"\n{'='*50}")
    print(f" AVERAGE EXECUTION TIME PER N OVER {NUM_RUNS} RUNS ")
    print(f"{'='*50}")
    for N_test in n_range:
        avg_seconds = np.mean(times_per_N[N_test])
        std_seconds = np.std(times_per_N[N_test])
        print(f"N = {N_test:2d} | Avg Time: {avg_seconds:.4f} s (±{std_seconds:.4f} s)")
    print(f"{'='*50}")
#------------------------------------------------------------------------------------------------------------------------------
        # ---- What was used in every run ----
    print(f"\n{'='*100}")
    print(f" RANDOM PULSE AND TIME STEPS USED IN EACH RUN (SEED = {SEED})")
    print(f"{'='*100}")
    for r, k, c1_, w1_, c2_, w2_ in run_log:
        print(f"Run {r:2d} | steps = {k:5d} | u(t) = {c1_:.4f}*sin({w1_:.4f}*t) + {c2_:.4f}*cos({w2_:.4f}*t)")

    # ---- Time for every N in every run ----
    print(f"\n{'='*100}")
    print(" TIME (seconds): one row per N, one column per run (steps in the header)")
    print(f"{'='*100}")
    print("  N | " + " | ".join(f"{entry[1]:>8d}" for entry in run_log))
    for N_test in n_range:
        print(f"{N_test:3d} | " + " | ".join(f"{t:8.3f}" for t in times_per_N[N_test]))



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.integrate import solve_ivp

# ---------- check that the data from the benchmark cell exists ----------
for needed in ("times_per_N", "run_log", "norms_per_N", "saved_states"):
    if needed not in globals():
        raise NameError(f"'{needed}' not found: add the 3 edits to the benchmark script and re-run it first.")
if not saved_states:
    raise RuntimeError("saved_states is empty: add the 3 edits to the benchmark script and re-run it first.")

# ---------- data ----------
n_list = list(n_range)
steps  = np.array([entry[1] for entry in run_log])      # steps used in each run
order  = np.argsort(steps)                              # sort runs by step count for line plots
colors = plt.cm.viridis(np.linspace(0, 0.9, len(n_list)))
T = 1.0

# =====================================================================
# GRAPH 1: time vs steps (log-log), one line per N  +  fixed/per-step cost fit
# =====================================================================
fig, (ax1) = plt.subplots(figsize=(15, 5))
for N, col in zip(n_list, colors):
    t = np.array(times_per_N[N])
    ax1.plot(steps[order], t[order], "o-", color=col, label=f"N={N}")
ax1.set_xscale("log"); ax1.set_yscale("log")
ax1.set_xlabel("number of Euler steps"); ax1.set_ylabel("time (s)")
ax1.set_title("Graph 1: time vs number of steps")
ax1.legend(ncol=2, fontsize=8); ax1.grid(True, which="both", alpha=0.3)

# =====================================================================
# GRAPH 2: time vs N (log y), one line per step count
# =====================================================================
fig, ax = plt.subplots(figsize=(8, 5))
cols = plt.cm.plasma(np.linspace(0, 0.9, len(steps)))
for rank, idx in enumerate(order):
    ax.plot(n_list, [times_per_N[N][idx] for N in n_list], "o-", color=cols[rank], label=f"{steps[idx]} steps")
ax.set_yscale("log")
ax.set_xlabel("N (number of NV centers)"); ax.set_ylabel("time (s)")
ax.set_title("Graph 2: time vs N")
ax.legend(fontsize=8, ncol=2); ax.grid(True, which="both", alpha=0.3)
plt.tight_layout(); plt.savefig("graph2_time_vs_N.png", dpi=150, bbox_inches="tight"); plt.show()

# =====================================================================
# GRAPH 3: the random pulse used in each run (and what Euler actually samples)
# =====================================================================
ncols = 5
nrows = int(np.ceil(len(run_log) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 3.2 * nrows))
axes = np.atleast_1d(axes).ravel()
t_fine = np.linspace(0, T, 2000)
for ax, (r, k, c1_, w1_, c2_, w2_) in zip(axes, run_log):
    ax.plot(t_fine, c1_ * np.sin(w1_ * t_fine) + c2_ * np.cos(w2_ * t_fine), color="tab:blue", lw=1.2)
    if k <= 200:
        t_i = np.arange(k) * (T / k)
        u_i = c1_ * np.sin(w1_ * t_i) + c2_ * np.cos(w2_ * t_i)
        ax.step(np.append(t_i, T), np.append(u_i, u_i[-1]), where="post", color="tab:red", lw=1.5)
        ax.plot(t_i, u_i, "k.", ms=4)
    else:
        ax.text(0.5, 0.04, f"{k} samples (too dense to draw)", transform=ax.transAxes, ha="center", fontsize=8)
    ax.set_title(f"Run {r} | {k} steps\nc1={c1_:.2f} w1={w1_:.1f} c2={c2_:.2f} w2={w2_:.1f}", fontsize=8)
    ax.grid(True, alpha=0.3)
for ax in axes[len(run_log):]:
    ax.axis("off")
fig.legend(handles=[Line2D([0], [0], color="tab:blue", lw=1.5, label="true pulse u(t)"),
                    Line2D([0], [0], color="tab:red", lw=1.5, label="what Euler uses (value at the start of each step)")],
           loc="lower center", ncol=2)
fig.suptitle("Graph 3: random pulse in each run", y=1.0)
plt.tight_layout(rect=(0, 0.05, 1, 1)); plt.savefig("graph3_pulses.png", dpi=150, bbox_inches="tight"); plt.show()

# =====================================================================
# GRAPH 4: final norm drift vs steps (log-log), one line per N
# =====================================================================
fig, ax = plt.subplots(figsize=(8, 5))
for N, col in zip(n_list, colors):
    drift = np.maximum(np.array(norms_per_N[N]) - 1.0, 1e-16)    # exact solution has norm 1, so drift should be 0
    ax.plot(steps[order], drift[order], "o-", color=col, label=f"N={N}")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("number of Euler steps"); ax.set_ylabel("final norm - 1   (0 = perfect)")
ax.set_title("Graph 4: norm drift vs number of steps")
ax.legend(ncol=2, fontsize=8); ax.grid(True, which="both", alpha=0.3)
plt.tight_layout(); plt.savefig("graph4_norm_vs_steps.png", dpi=150, bbox_inches="tight"); plt.show()

# =====================================================================
# GRAPH 5: error vs an accurate reference solution (small N only)
# =====================================================================
def reference_state(N, c1_, w1_, c2_, w2_):
    params = {'N': N, 'D': [2870.0] * N, 'omega_0': 2870.0, 'gamma_e': 28000.0,
              'Bz': [1e-3] * N, 'zeta': np.pi / 4, 'g': [0.1] * (N - 1)}
    system = NVCenterNearestNeighborGPU(params)
    H0 = system.get_drift_hamiltonian()
    Hc = system.get_control_hamiltonian()
    psi0 = system.get_ground_state()
    u = lambda t: c1_ * np.sin(w1_ * t) + c2_ * np.cos(w2_ * t)
    f = lambda t, y: -1j * (H0 @ y + u(t) * (Hc @ y))
    sol = solve_ivp(f, [0, T], psi0, method="DOP853", rtol=1e-10, atol=1e-12)
    return sol.y[:, -1]

errors = {}
for N in [n for n in n_list if n <= 4]:
    errs = []
    for (r, k, c1_, w1_, c2_, w2_) in run_log:
        ref = reference_state(N, c1_, w1_, c2_, w2_)
        got = saved_states[(N, r)]
        errs.append(np.linalg.norm(got - ref) / np.linalg.norm(ref))
    errors[N] = np.array(errs)

fig, ax = plt.subplots(figsize=(8, 5))
for N, e in errors.items():
    ax.plot(steps[order], e[order], "o-", color=colors[n_list.index(N)], label=f"N={N}")
first = next(iter(errors))
anchor_k, anchor_e = steps[order][-1], errors[first][order][-1]
guide_k = np.array([anchor_k / 100, anchor_k])
ax.plot(guide_k, anchor_e * anchor_k / guide_k, "k--", label="slope -1 (error proportional to 1/steps)")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("number of Euler steps"); ax.set_ylabel("relative error vs accurate solution")
ax.set_title("Graph 5: error vs number of steps")
ax.legend(fontsize=8); ax.grid(True, which="both", alpha=0.3)
plt.tight_layout(); plt.savefig("graph5_error_vs_steps.png", dpi=150, bbox_inches="tight"); plt.show()

print("Saved: graph1_time_vs_steps.png, graph2_time_vs_N.png, graph3_pulses.png, graph4_norm_vs_steps.png, graph5_error_vs_steps.png")